## Performance Tuning with `OPTIMIZE` File Compaction

### Create Raw Dataset (Simulate Real Data)

In [0]:
from pyspark.sql.functions import *
import random

data = []

cities = ["London", "Manchester", "Birmingham", "Leeds"]
segments = ["Retail", "Premium", "Business"]

for i in range(100000):
    data.append((
        f"CUST-{i}",
        random.choice(cities),
        random.choice(segments),
        random.randint(100, 10000),
        random.randint(1, 10)
    ))

df = spark.createDataFrame(data, [
    "customer_id", "city", "segment", "revenue", "transactions"
])

display(df)

### Create the Small File Problem

In [0]:
df.repartition(200).write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("customer_sales")

### Observe the File Structure

In [0]:
%sql
DESCRIBE DETAIL default.customer_sales;

### Run Baseline Query (Slow Performance)

In [0]:
%sql
SELECT city, SUM(revenue)
FROM customer_sales
WHERE city = 'London'
GROUP BY city;

### OPTIMIZE Command (File Compaction)

In [0]:
%sql
OPTIMIZE default.customer_sales;

### Description of the Table After OPTIMIZE

In [0]:
%sql
DESCRIBE DETAIL default.customer_sales;

### Run Query After Optimize (Improved Performance)

In [0]:
%sql
SELECT city, SUM(revenue)
FROM customer_sales
WHERE city = 'London'
GROUP BY city;

### Time Travel Before VACUUM (Files Still Present)

In [0]:
%sql
SELECT * FROM customer_sales VERSION AS OF 0;

### Disable Retention Period Check for Testing

In [0]:
%sql
SET spark.databricks.delta.retentionDurationCheck.enabled = false;

### Run the VACUUM Command (Remove Old Files)

In [0]:
%sql
VACUUM customer_sales RETAIN 0 HOURS;

### Time Travel After VACUUM (Files Removed, Query Fails)

In [0]:
%sql
SELECT * FROM customer_sales VERSION AS OF 0;